# Projeto Andromeda-Net: Coleta e Limpeza de Dados (Fórmula 1)

Este notebook realiza a extração e o pré-processamento dos dados do dataset **Formula 1 World Championship**. O objetivo é preparar os dados para modelagem de redes complexas, cumprindo os requisitos de **Reprodutibilidade** e **Validação de ETL** exigidos nas regras do projeto (sem dados 'hardcoded' e tratamento adequado de nulos).

In [1]:
import pandas as pd
import numpy as np
import os

# Configuração de caminhos baseada na estrutura do projeto
RAW_DATA_DIR = '../../data/'
PROCESSED_DATA_DIR = '../../data/processed/'

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)

## 1. Carregamento dos Dados Brutos

Selecionando tabelas fundamentais para redes bipartite (Pilotos-Construtores) ou redes de co-ocorrência/teammates.

In [2]:
# Carregamento
df_drivers = pd.read_csv(os.path.join(RAW_DATA_DIR, 'drivers.csv'))
df_constructors = pd.read_csv(os.path.join(RAW_DATA_DIR, 'constructors.csv'))
df_races = pd.read_csv(os.path.join(RAW_DATA_DIR, 'races.csv'))
df_results = pd.read_csv(os.path.join(RAW_DATA_DIR, 'results.csv'))

print(f'Pilotos: {df_drivers.shape[0]}')
print(f'Construtores: {df_constructors.shape[0]}')
print(f'Corridas: {df_races.shape[0]}')
print(f'Resultados (Links potenciais): {df_results.shape[0]}')

Pilotos: 861
Construtores: 212
Corridas: 1125
Resultados (Links potenciais): 26759


## 2. Limpeza (Tratamento de Nulos)

O dataset Kaggle utiliza `\N` para valores nulos. Isso quebra cálculos. Vamos corrigir e tipar corretamente.

In [3]:
# Tratando \N para NaN
df_results.replace(r'\\N', np.nan, regex=True, inplace=True)

cols_num = ['position', 'points', 'laps', 'milliseconds', 'fastestLap', 'rank']
for col in cols_num:
    if col in df_results.columns:
        df_results[col] = pd.to_numeric(df_results[col], errors='coerce')

## 3. Merge e Criação da Base do Grafo

Para evitar joins futuros, criaremos uma tabela unificada contendo `(Piloto, Construtor, Corrida, Ano)`.

In [4]:
# Merge de Results e Races
df_merged = df_results.merge(df_races[['raceId', 'year', 'name']], on='raceId', how='left')

# Preparar e dar Merge em Drivers
df_drivers['driver_name'] = df_drivers['forename'] + ' ' + df_drivers['surname']
df_merged = df_merged.merge(df_drivers[['driverId', 'driverRef', 'driver_name', 'nationality']], on='driverId', how='left')

# Merge em Constructors
df_merged = df_merged.merge(df_constructors[['constructorId', 'constructorRef', 'name']], 
                            on='constructorId', how='left', suffixes=('_race', '_constructor'))

df_merged.rename(columns={'name_constructor': 'constructor_name', 'name_race': 'race_name'}, inplace=True)

# Seleção de Colunas
cols_grafo = ['year', 'raceId', 'race_name', 'driverId', 'driver_name', 'nationality', 'constructorId', 'constructor_name', 'grid', 'position', 'points']
df_graph_base = df_merged[cols_grafo]

df_graph_base.head()

,year,raceId,race_name,driverId,driver_name,nationality,constructorId,constructor_name,grid,position,points
0,2008,18,Australian Grand Prix,1,Lewis Hamilton,British,1,McLaren,1,1.0,10.0
1,2008,18,Australian Grand Prix,2,Nick Heidfeld,German,2,BMW Sauber,5,2.0,8.0
2,2008,18,Australian Grand Prix,3,Nico Rosberg,German,3,Williams,7,3.0,6.0
3,2008,18,Australian Grand Prix,4,Fernando Alonso,Spanish,4,Renault,11,4.0,5.0
4,2008,18,Australian Grand Prix,5,Heikki Kovalainen,Finnish,1,McLaren,3,5.0,4.0


## 4. Salvando Dataset Processado

O arquivo final será utilizado nas etapas de EDA e construção da rede em NetworkX.

In [5]:
output = os.path.join(PROCESSED_DATA_DIR, 'f1_graph_base.csv')
df_graph_base.to_csv(output, index=False)
print('ETL Concluído. Dados salvos com sucesso!')

ETL Concluído. Dados salvos com sucesso!
